# 05 — Explain Mechanism

EXPLAIN — the mechanistic backbone (theory-anchored on Minority Collapse). Hypothesis tournament + neural-collapse geometry + per-family proximate causes. The surviving mechanism DEFINES the diagnostic's feature set.

In [1]:
# ── FULL bootstrap — run FIRST ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
      "| split:", CFG['split']['primary'])

Mounted at /content/drive
GPU: Tesla T4 | split: temporal_within_capture


In [2]:
# --- Colab bootstrap (config-driven; never hardcode a path) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds, require_frozen
set_all_seeds(CFG['anchor_seed'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
from src import data
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df), "| test:", len(splits["test"]))

rows: 3661696 | test: 549271


In [5]:
import sys
for m in list(sys.modules):
    if m.startswith("src"): del sys.modules[m]   # clear stale cache

try:
    import src.explain
    print("explain imported OK")
except Exception as e:
    import traceback; traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_10661/4097806076.py", line 6, in <cell line: 0>
    import src.explain
ModuleNotFoundError: No module named 'src.explain'


In [6]:
import os
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
print("explain.py exists:", os.path.exists(f"{REPO}/src/explain.py"))
print("\nfiles in src/:")
for f in sorted(os.listdir(f"{REPO}/src")):
    p = f"{REPO}/src/{f}"
    if os.path.isfile(p):
        print(f"  {f}  ({os.path.getsize(p)} bytes)")
print("\nimporting src from:", os.path.dirname(__import__('src').__file__))

explain.py exists: True

files in src/:
  __init__.py  (0 bytes)
  compression.py  (10934 bytes)
  config.py  (4438 bytes)
  crux.py  (1161 bytes)
  data.py  (10577 bytes)
  diagnostic.py  (1892 bytes)
  explain.py  (5578 bytes)
  geometry.py  (4141 bytes)
  metrics.py  (4458 bytes)
  mitigate.py  (1639 bytes)
  models.py  (3897 bytes)
  train.py  (7539 bytes)

importing src from: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src


In [8]:
import importlib.util, sys
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
spec = importlib.util.spec_from_file_location("src.explain", f"{REPO}/src/explain.py")
explain = importlib.util.module_from_spec(spec)
sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
print("explain loaded:", hasattr(explain, "per_class_geometry"))

explain loaded: True


In [9]:
import importlib
from src import data, train, compression, models
for m in (data, train, compression): importlib.reload(m)
import pandas as pd, numpy as np, torch

# 1) reload M0 anchor, rebuild prune80
anchor, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",
                                                  CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
prune80, _le, _sc = compression.prune_and_finetune(anchor, df, "ciciot2023", splits,
                                                   CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
print("M0 + prune80 ready")

# 2) extract penultimate features from BOTH models
fM0, lM0, y = explain.extract_features(anchor, df, splits, scaler, feat_cols, le)
fP8, lP8, _ = explain.extract_features(prune80, df, splits, scaler, feat_cols, le)
print("features:", fM0.shape, "->", fP8.shape)

# 3) per-class geometry on M0 + prune80 with bootstrap CIs
geomM0 = explain.per_class_geometry(fM0, lM0, y, le, n_boot=200, seed=0)
geomP8 = explain.per_class_geometry(fP8, lP8, y, le, n_boot=200, seed=0)

# 4) THE TEST: does baseline geometry predict prune80 Δrecall?
delta = pd.read_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"), index_col=0)["prune80"]
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
meas = tiers[tiers=="measurable"].index

corr = explain.correlate_geometry_vs_collapse(geomM0.loc[meas], delta.loc[meas], "mean_cos_to_others")
print("\n=== HYPOTHESIS TEST: baseline geometry vs prune80 collapse (measurable tier) ===")
print({k:(round(v,4) if isinstance(v,float) else v) for k,v in corr.items()})

corr_m = explain.correlate_geometry_vs_collapse(geomM0.loc[meas], delta.loc[meas], "margin")
print("margin vs collapse:", {k:(round(v,4) if isinstance(v,float) else v) for k,v in corr_m.items()})

# 5) per-class geometry next to collapse, worst first
R = pd.read_csv(PATHS.tables("compression","cnn1d_per_class_recall_matrix.csv"),index_col=0)
tbl = pd.DataFrame({
    "M0_recall": R["M0"],
    "prune80_d_recall": delta,
    "M0_geom_cos": geomM0["mean_cos_to_others"],
    "M0_margin": geomM0["margin"],
    "prune80_geom_cos": geomP8["mean_cos_to_others"],
    "tier": tiers,
}).loc[meas].sort_values("prune80_d_recall")
print("\n=== per-class: baseline geometry vs collapse (worst first) ===")
print(tbl.round(3).to_string())

geomM0.to_csv(PATHS.tables("explain","cnn1d_M0_geometry.csv"))
geomP8.to_csv(PATHS.tables("explain","cnn1d_prune80_geometry.csv"))
tbl.to_csv(PATHS.tables("explain","geometry_vs_collapse.csv"))
print("\nsaved geometry tables -> results/tables/explain/")

M0 + prune80 ready
features: (549271, 128) -> (549271, 128)

=== HYPOTHESIS TEST: baseline geometry vs prune80 collapse (measurable tier) ===
{'n': 13, 'spearman_r': 0.1923, 'spearman_p': 0.5291, 'kendall_tau': 0.1538, 'kendall_p': 0.5098, 'interpretation': 'no/positive relationship'}
margin vs collapse: {'n': 13, 'spearman_r': -0.2637, 'spearman_p': 0.3839, 'kendall_tau': -0.1795, 'kendall_p': 0.4354, 'interpretation': 'fragile baseline geometry predicts collapse'}

=== per-class: baseline geometry vs collapse (worst first) ===
                      M0_recall  prune80_d_recall  M0_geom_cos  M0_margin  prune80_geom_cos        tier
label                                                                                                  
DoS-UDP_Flood             0.682            -0.682       -0.126      1.271            -0.182  measurable
DoS-HTTP_Flood            0.867            -0.654       -0.022      2.142             0.011  measurable
Recon-HostDiscovery       0.705            -0.629

In [10]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def confusability_reallocation(model_M0, model_comp, df, splits, scaler, feat_cols, le,
                               collapsed_classes, is_half_comp=False, which="test"):
    """Test the CONFUSABILITY mechanism: when a class collapses under compression,
    where do its samples go? For each collapsed class, find which OTHER class
    absorbs the most of its (now-misclassified) test samples, and whether that
    absorber's recall RISES. If collapse pairs with a sibling's gain, the
    mechanism is winner-take-all reallocation among confusable classes."""
    import numpy as np, pandas as pd
    fM0, lM0, y = extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    fC, lC, _ = extract_features(model_comp, df, splits, scaler, feat_cols, le,
                                 which=which, is_half=is_half_comp)
    predM0 = lM0.argmax(1); predC = lC.argmax(1)
    name = lambda i: le.classes_[int(i)]
    idx = {n: i for i, n in enumerate(le.classes_)}
    def recall(pred):
        return {int(c): float((pred[y == c] == c).mean()) if (y == c).any() else np.nan
                for c in np.unique(y)}
    rM0, rC = recall(predM0), recall(predC)
    rows = []
    for cls in collapsed_classes:
        c = idx[cls]
        mask = (y == c)
        comp_pred = predC[mask]
        wrong = comp_pred[comp_pred != c]
        if len(wrong) == 0:
            rows.append({"collapsed_class": cls, "top_absorber": None}); continue
        vals, counts = np.unique(wrong, return_counts=True)
        top = vals[counts.argmax()]
        frac = counts.max() / mask.sum()
        rows.append({
            "collapsed_class": cls,
            "M0_recall": round(rM0[c], 3),
            "comp_recall": round(rC[c], 3),
            "top_absorber": name(top),
            "absorber_took_frac": round(float(frac), 3),
            "absorber_recall_M0": round(rM0[int(top)], 3),
            "absorber_recall_comp": round(rC[int(top)], 3),
            "absorber_recall_gain": round(rC[int(top)] - rM0[int(top)], 3),
        })
    return pd.DataFrame(rows).sort_values("absorber_took_frac", ascending=False)
'''
with open(f"{REPO}/src/explain.py","a") as f:
    f.write(ADD)
print("appended confusability_reallocation")

appended confusability_reallocation


In [11]:
# reload explain from file to pick up the new function (file-path bypass)
import importlib.util, sys
spec = importlib.util.spec_from_file_location("src.explain",
        "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
import pandas as pd

# the 10 classes that collapsed under prune80 (from the collapse flags)
flags = pd.read_csv(PATHS.tables("compression","cnn1d_collapse_flags.csv"), index_col=0)
collapsed = list(flags.index[flags["prune80"]])
print("collapsed under prune80:", collapsed, "\n")

conf = explain.confusability_reallocation(anchor, prune80, df, splits, scaler, feat_cols, le, collapsed)
print("=== CONFUSABILITY: where did each collapsed class's samples go under prune80? ===")
print(conf.to_string(index=False))
conf.to_csv(PATHS.tables("explain","prune80_confusability.csv"), index=False)
print("\nsaved -> results/tables/explain/prune80_confusability.csv")

collapsed under prune80: ['BenignTraffic', 'BrowserHijacking', 'CommandInjection', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SlowLoris', 'DDoS-TCP_Flood', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Mirai-greeth_flood', 'Mirai-greip_flood', 'Mirai-udpplain', 'Recon-HostDiscovery', 'Recon-OSScan', 'Recon-PortScan', 'SqlInjection', 'XSS'] 

=== CONFUSABILITY: where did each collapsed class's samples go under prune80? ===
     collapsed_class  M0_recall  comp_recall            top_absorber  absorber_took_frac  absorber_recall_M0  absorber_recall_comp  absorber_recall_gain
       DoS-SYN_Flood      0.164        0.000          DDoS-SYN_Flood               0.973               0.523                 0.940                 0.417
       DoS-UDP_Flood      0.682        0.000          DDoS-UDP_Flood               0.956               0.742                 0.997                 0.255
    CommandInjection      0

In [12]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git add -A
!git commit -q -m "EXPLAIN: simple geometry hypothesis NULL (Spearman +0.19/-0.26, ns); confusability test reveals two-regime collapse — protocol-confusable DoS/DDoS pairs consolidate onto sibling, AND 10 hard/sparse classes collapse into single absorber sink (VulnerabilityScan). Mechanism is structured sink-consolidation, not Minority Collapse"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@914c5b17749d.(none)')
 Everything up-to-date



In [ ]:
# Computes trust metrics — refuse to run until the prereg is frozen.
require_frozen()

## Neural-collapse geometry (the theory layer)

Per-class ETF deviation + minority-collapse angle at baseline and under each compression level. Rare classes start with least angular margin -> fail first.

In [ ]:
from src import geometry
# etf = geometry.etf_deviation(feats, labels)
# mci = geometry.minority_collapse_index(feats, labels)
# margins = geometry.per_class_margin(logits, labels)

## Tournament + per-family causes + rarity-vs-separability ablation (D4)

Dissociate pruning vs quantisation vs distillation per class; subsample a frequent class to a rare class's count to separate rarity from separability.

In [ ]:
# H1 info-loss(probe) / H2 decision-rule(margin) / H3 optimisation(PTQ vs finetune) / H4 rarity-vs-sep

## Causal-claim checkpoint

The geometry must predict collapse CONSISTENTLY across all three architectures, else withdraw the causal claim to correlational.

In [ ]:
# record cross-architecture consistency of geometry->collapse

## End-of-unit (non-negotiable)

In [ ]:
# --- End-of-unit discipline (run before moving on) ---
# 1) outputs saved to Drive  2) commit + push  3) push CSVs/figures  4) confirm sync
# !cd $REPO && git add -A && git commit -m 'nb05: <meaningful message>' && git push
print('Saved under:', PATHS.tables().parent)
